# 🪝 Notebook 8: Webhooks

So far we've focused on the **client → server** hop (polling, SSE, WebSockets).
**Webhooks** flip the direction on the *server-to-server* side: instead of your
server polling a vendor's API ("any new orders yet?"), the vendor **POSTs an
HTTP request to your server** the moment an event happens.

```
Polling:                              Webhooks:
┌────────┐  HTTP GET every 10s         ┌────────┐  HTTP POST when it happens
│  you   │ ───────────────────▶        │ vendor │ ────────────────────────▶
│        │                             │        │        to your URL
│        │ ◀───── 99% empty ─────      └────────┘
└────────┘
```

### Where you already rely on webhooks
| Service | Event | What your server receives |
|---|---|---|
| Stripe | `payment_intent.succeeded` | order is paid, ship it |
| GitHub | `push`, `pull_request` | trigger CI build |
| Slack | slash command / message | respond in channel |
| Twilio | `incoming_sms` | reply to SMS |
| Shopify | `orders/create` | update inventory |

### What we'll build
1. Start a small **receiver** in this notebook (your server) — with signature
   verification, replay protection and idempotency built in from line one.
2. Register its URL with a **vendor** server (`webhook_server.py`).
3. Watch events get POSTed to us in real time.
4. **Attack our own receiver** four ways and watch every attack get refused.
5. Simulate a flaky receiver and see the vendor **retry with backoff**.
6. Deliver the same event three times and ship the order exactly **once**.
7. Compare webhooks vs polling for a concrete workload.


## ⚙️ Step 1 — start the vendor server

The next cell starts `servers/webhook_server.py` for you and shuts it down
when the kernel exits. To watch its delivery log live instead, start it
yourself in a second terminal first — the notebook leaves an already-listening
port alone:

```bash
python servers/webhook_server.py   # listens on http://localhost:5005
```

It exposes `/subscriptions`, `/trigger`, `/deliveries`, `/secret`, `/health`.

In [ ]:
# Start the webhook vendor server this notebook talks to.
#
# `ensure_server` is idempotent: if you already started the server by hand
# in another terminal it is left alone, otherwise it is launched as a
# background process using this notebook's own interpreter (the lab .venv)
# and shut down when the kernel exits. This is what makes the notebook
# runnable on its own -- previously the next cell just died with a raw
# ConnectionError if you had not started the server first.
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "servers"))
from lab_servers import ensure_server

print(ensure_server(5005))

import requests

health = requests.get("http://localhost:5005/health", timeout=5)
health.raise_for_status()
print("health:", health.json())


## 📬 Step 2 — run a receiver inside the notebook

A webhook receiver is just an HTTP endpoint that accepts POSTs — but "just an
endpoint" is exactly how people get burned. **Your webhook URL is a public,
unauthenticated write endpoint on the internet.** Anyone who guesses it can
POST `{"event": "payment.succeeded"}` at you, and a receiver that trusts the
body will happily ship the goods.

So we build the receiver with its three defences in place from the first line,
rather than bolting them on later:

| Defence | What it stops | Where it lives below |
|---|---|---|
| **HMAC signature check** | Forged events from anyone who isn't the vendor | `verify_signature()` |
| **Timestamp freshness** | Replay of a *genuine* event captured off the wire | `MAX_AGE_SECONDS` |
| **Dedupe on `delivery_id`** | Double-charging when the vendor retries | `processed` ledger |

We'll use FastAPI + uvicorn on port **5006** so the vendor has somewhere to
deliver to.

In [ ]:
import hashlib
import hmac
import json
import socket
import threading
import time

import requests
import uvicorn
from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse

VENDOR = "http://localhost:5005"

# In real life this comes from your secrets manager and matches what the
# vendor showed you once in their dashboard. The lab server exposes it on
# /secret purely so this notebook is self-contained.
SECRET = requests.get(f"{VENDOR}/secret", timeout=5).json()["secret"]
MAX_AGE_SECONDS = 300          # reject anything older than 5 minutes

received = []                  # deliveries we accepted
rejected = []                  # deliveries we refused, and why
processed: dict[str, dict] = {}  # delivery_id -> result  (the idempotency ledger)
orders_shipped: list[dict] = []  # the irreversible side effect we are protecting

receiver = FastAPI()


def verify_signature(body: bytes, timestamp: str | None, header: str | None) -> str | None:
    """Return None if the request is authentic, else the reason it isn't."""
    if not header or not timestamp:
        return "missing signature headers"
    try:
        age = time.time() - int(timestamp)
    except ValueError:
        return "unparseable timestamp"
    if abs(age) > MAX_AGE_SECONDS:
        # Without this, an attacker who captures ONE valid delivery can replay
        # it forever -- the signature stays valid because the body never changes.
        return f"stale timestamp ({age:.0f}s old)"
    expected = "sha256=" + hmac.new(
        SECRET.encode(), f"{timestamp}.".encode() + body, hashlib.sha256
    ).hexdigest()
    # compare_digest is constant-time: a plain `==` leaks how many leading
    # bytes were right, which is enough to forge a signature byte by byte.
    if not hmac.compare_digest(expected, header):
        return "bad signature"
    return None


@receiver.post("/hook")
async def hook(request: Request):
    body = await request.body()
    reason = verify_signature(
        body,
        request.headers.get("x-webhook-timestamp"),
        request.headers.get("x-webhook-signature"),
    )
    if reason:
        rejected.append({"reason": reason, "body": body.decode()})
        print(f"🚫 rejected webhook: {reason}")
        # 401, not 500 -- and note we do NOT retry-bait the vendor with a 5xx.
        return JSONResponse({"error": reason}, status_code=401)

    delivery_id = request.headers.get("x-webhook-delivery-id", "")
    if delivery_id in processed:
        # The vendor retried something we already handled. ACK it again so it
        # stops retrying, but do NOT redo the work.
        print(f"🔁 duplicate delivery {delivery_id[:8]} - already handled, ACKing again")
        return {"ok": True, "duplicate": True, **processed[delivery_id]}

    payload = json.loads(body)
    # The irreversible bit. In a real system: capture the payment, ship the
    # order, send the email. Everything above exists to protect this line.
    orders_shipped.append(payload)
    processed[delivery_id] = {"order": payload["data"].get("id")}

    received.append({
        "headers": dict(request.headers),
        "body": body.decode(),
        "delivery_id": delivery_id,
        "at": time.time(),
    })
    return {"ok": True, "duplicate": False}


@receiver.post("/flaky")
async def flaky(request: Request):
    """Fails twice, then succeeds — used to demo the vendor's retry schedule."""
    body = await request.body()
    reason = verify_signature(
        body,
        request.headers.get("x-webhook-timestamp"),
        request.headers.get("x-webhook-signature"),
    )
    if reason:
        return JSONResponse({"error": reason}, status_code=401)

    delivery_id = request.headers.get("x-webhook-delivery-id", "")
    # Count attempts PER DELIVERY, not globally -- a retry re-sends the same
    # delivery_id, which is exactly what makes the count meaningful.
    attempt = len([d for d in received if d.get("delivery_id") == delivery_id]) + 1
    received.append({"endpoint": "flaky", "delivery_id": delivery_id,
                     "attempt": attempt, "body": body.decode(), "at": time.time()})
    if attempt < 3:
        # 503 says "try me again"; a 4xx would tell the vendor to give up.
        return JSONResponse({"error": "temporary"}, status_code=503)
    return {"ok": True, "accepted_on_attempt": attempt}


def _listening(port: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.3)
        return s.connect_ex(("127.0.0.1", port)) == 0


# Idempotent start, so re-running this cell doesn't raise "address already in
# use" inside a background thread where you'd never see the traceback.
if not _listening(5006):
    threading.Thread(
        target=lambda: uvicorn.run(receiver, host="127.0.0.1", port=5006, log_level="warning"),
        daemon=True,
    ).start()
    for _ in range(40):
        if _listening(5006):
            break
        time.sleep(0.2)

assert _listening(5006), "receiver failed to start on port 5006"
print("✅ Receiver listening on http://127.0.0.1:5006")
print(f"   signature: HMAC-SHA256 over '<timestamp>.<body>', max age {MAX_AGE_SECONDS}s")

## 🔗 Step 3 — register our URL with the vendor

This is the typical "add webhook" step you do in a SaaS dashboard.


In [ ]:
MY_URL = "http://127.0.0.1:5006/hook"

# Tidy up any leftover subscriptions from a previous run
for s in requests.get(f"{VENDOR}/subscriptions", timeout=5).json()["subscriptions"]:
    requests.delete(f"{VENDOR}/subscriptions/{s['id']}", timeout=5)

resp = requests.post(f"{VENDOR}/subscriptions", json={"url": MY_URL}, timeout=5)
sub = resp.json()
print("📌 Registered:", sub)

## ⚡ Step 4 — trigger some events

Ask the vendor to send us a couple of events. Because we're the only
subscriber, both should arrive at our `/hook`.


In [ ]:
received.clear()
rejected.clear()
processed.clear()
orders_shipped.clear()

for payload in [
    {"event": "order.created", "data": {"id": 42, "amount": 19.99}},
    {"event": "order.paid",    "data": {"id": 42, "amount": 19.99}},
]:
    r = requests.post(f"{VENDOR}/trigger", json=payload, timeout=10)
    print("→ trigger:", payload["event"], r.json())

# Give the event loop a moment to drain
time.sleep(0.3)

print(f"\n📥 Received {len(received)} webhook call(s):")
for rec in received:
    data = json.loads(rec['body'])
    print(f"  • {data['event']} (delivery_id={data['delivery_id']})")

# Both events must have made it through the signature check. If the receiver
# silently rejected them we'd see "Received 0" and a wall of 🚫 above.
assert len(received) == 2, f"expected 2 deliveries, got {len(received)}"
assert not rejected, f"the vendor's own deliveries were rejected: {rejected}"
assert len({r["delivery_id"] for r in received}) == 2, "delivery ids must be unique per event"

## 🔐 Step 5 — the signature check we already installed

**Never trust a webhook blindly.** Anyone on the internet can POST to your URL.
The vendor signs each delivery with a shared secret; the receiver recomputes
the HMAC and compares.

Two details that people get wrong:

- **What gets signed.** Signing the body alone means a captured request stays
  valid forever — an attacker replays the same bytes and the signature still
  checks out. Stripe signs `"<timestamp>.<payload>"` and receivers reject
  anything older than a few minutes; our vendor and receiver do the same, via
  the `X-Webhook-Timestamp` header.
- **How you compare.** `hmac.compare_digest`, never `==`. A normal string
  comparison returns early on the first mismatched byte, and that timing
  difference is enough to reconstruct a valid signature one byte at a time.

This is the same shape as Stripe's `Stripe-Signature` and GitHub's
`X-Hub-Signature-256`.

The receiver in Step 2 already enforces all of this. Let's prove it works —
by attacking it.

In [ ]:
# Prove the receiver's defences actually fire. Each attack must be REFUSED;
# if any of them returns 200 the receiver has a real vulnerability.

genuine = received[0]
body = genuine["body"].encode()
ts = genuine["headers"]["x-webhook-timestamp"]
sig = genuine["headers"]["x-webhook-signature"]

def attack(name, body_bytes, headers, expect_status):
    r = requests.post("http://127.0.0.1:5006/hook", data=body_bytes,
                      headers={"Content-Type": "application/json", **headers}, timeout=5)
    if r.status_code != expect_status:
        verdict = "❌ WRONG   "
    else:
        verdict = "✅ refused " if expect_status == 401 else "✅ accepted"
    print(f"{verdict} {name:34} → {r.status_code} {r.json().get('error', '')}")
    return r.status_code

# 1. No signature at all -- the naive "it's just an HTTP endpoint" receiver.
s1 = attack("no signature header", body,
            {}, 401)

# 2. Body tampered, original signature reused. £19.99 becomes £0.01.
tampered = genuine["body"].replace("19.99", "0.01").encode()
s2 = attack("tampered body, real signature", tampered,
            {"X-Webhook-Timestamp": ts, "X-Webhook-Signature": sig}, 401)

# 3. Signature forged with the wrong secret.
forged = "sha256=" + hmac.new(b"attacker-guessed-secret",
                              f"{ts}.".encode() + body, hashlib.sha256).hexdigest()
s3 = attack("signature from the wrong secret", body,
            {"X-Webhook-Timestamp": ts, "X-Webhook-Signature": forged}, 401)

# 4. REPLAY: a perfectly genuine, correctly-signed request captured off the
#    wire and re-sent an hour later. Body-only signing would let this through.
old_ts = str(int(time.time()) - 3600)
old_sig = "sha256=" + hmac.new(SECRET.encode(),
                               f"{old_ts}.".encode() + body, hashlib.sha256).hexdigest()
s4 = attack("valid signature, 1 hour old", body,
            {"X-Webhook-Timestamp": old_ts, "X-Webhook-Signature": old_sig}, 401)

# 5. Control: a correctly signed, fresh request must still be ACCEPTED --
#    otherwise we have "secured" the endpoint by breaking it.
fresh_ts = str(int(time.time()))
fresh_body = json.dumps({"event": "order.created", "data": {"id": 99},
                         "delivery_id": "control-1", "timestamp": "now"}).encode()
fresh_sig = "sha256=" + hmac.new(SECRET.encode(),
                                 f"{fresh_ts}.".encode() + fresh_body, hashlib.sha256).hexdigest()
s5 = attack("correctly signed, fresh (control)", fresh_body,
            {"X-Webhook-Timestamp": fresh_ts, "X-Webhook-Signature": fresh_sig,
             "X-Webhook-Delivery-Id": "control-1"}, 200)

assert s1 == s2 == s3 == 401, "a forged or tampered webhook was accepted"
assert s4 == 401, "a replayed (genuine but old) webhook was accepted -- sign the timestamp"
assert s5 == 200, "a legitimate webhook was rejected -- the check is too strict"
print("\n🔐 4 attacks refused, legitimate traffic still accepted.")

## 🔁 Step 6 — retries with exponential backoff

Receivers are flaky: a deploy, a full disk, a lock timeout. Good webhook
vendors retry failed deliveries with a growing gap between attempts —
**exponential backoff**, so a receiver that is struggling isn't hammered while
it recovers. Our vendor waits `0.5s`, then `1.0s`, then gives up after
3 attempts.

> Production vendors add **jitter** (a random ± on each gap). Without it, every
> receiver that failed during the same outage retries in lockstep and
> re-creates the stampede. We leave jitter off here only so the timings below
> are reproducible.

Let's point the vendor at our `/flaky` endpoint, which returns **503** on the
first two attempts and **200** on the third.

In [ ]:
# Swap the subscription URL to the flaky endpoint
for s in requests.get(f"{VENDOR}/subscriptions", timeout=5).json()["subscriptions"]:
    requests.delete(f"{VENDOR}/subscriptions/{s['id']}", timeout=5)

requests.post(f"{VENDOR}/subscriptions", json={"url": "http://127.0.0.1:5006/flaky"}, timeout=5)

# Remember how many attempts the vendor had logged BEFORE this trigger, so we
# report this delivery's attempts and not the whole session's history.
before = requests.get(f"{VENDOR}/deliveries", timeout=5).json()["total"]

received.clear()
start = time.time()
requests.post(f"{VENDOR}/trigger",
              json={"event": "order.created", "data": {"id": 7}},
              timeout=30)
elapsed = time.time() - start
print(f"\n⏱️  Total vendor time: {elapsed:.2f}s")

log = requests.get(f"{VENDOR}/deliveries", params={"limit": 50}, timeout=5).json()
attempts = log["deliveries"][before:]

print("\n📊 Vendor-side delivery attempts for THIS trigger:")
for d in attempts:
    print(f"  attempt {d['attempt']} → status {d.get('status_code')} "
          f"({'ok' if d['ok'] else 'FAIL'})  delivery_id={d['delivery_id'][:8]}")

statuses = [d["status_code"] for d in attempts]
ids = {d["delivery_id"] for d in attempts}

print(f"\n💡 Same delivery_id on all {len(attempts)} attempts: {len(ids) == 1}")
print("   That is what makes the retry deduplicable on the receiving end --")
print("   a vendor that minted a fresh id per attempt would be unfixable.")

assert statuses == [503, 503, 200], (
    f"expected the vendor to retry through 503,503 to 200, got {statuses}"
)
assert len(ids) == 1, (
    f"the vendor changed delivery_id between retries ({ids}) -- the receiver "
    f"then has no way to tell a retry from a new event"
)
# 0.5s + 1.0s of backoff must actually have been slept, otherwise "exponential
# backoff" is just a comment.
assert elapsed >= 1.5, (
    f"the whole retry schedule took {elapsed:.2f}s -- the backoff waits are "
    f"not happening"
)
print(f"   Backoff really slept: {elapsed:.2f}s ≥ 0.5s + 1.0s.")

## 🧊 Step 6b — idempotency: surviving the retry

Retries are the vendor doing its job. But look at what a retry means for you:
the **same event arrives more than once**, and the second copy is
byte-identical — same body, same signature, same `delivery_id`. Every defence
from Step 5 waves it straight through, because it *is* genuine.

The only thing standing between a retry and a double shipment is a
**ledger keyed on `delivery_id`**: before doing anything irreversible, check
whether you already did it.

```
POST /hook  delivery_id=abc123   → not seen → ship order → record abc123 → 200
POST /hook  delivery_id=abc123   → SEEN     → skip work  → 200 (again)
```

Note the second response is still a **200**. Answering 409 or 500 to a
duplicate just makes the vendor retry it again, forever.

Why duplicates happen even when nothing is broken: your handler ships the
order, then the ACK is lost on the way back. The vendor never heard a 200, so
it retries — correctly. There is no protocol that removes this case; the
receiver has to be idempotent.

In [ ]:
# Deliver the SAME event three times, exactly as a retrying vendor would:
# identical body, identical signature, identical delivery_id.

orders_shipped.clear()
processed.clear()

payload = json.dumps({"event": "order.paid", "data": {"id": 4242, "amount": 99.0},
                      "delivery_id": "dup-demo-1", "timestamp": "now"}).encode()
ts = str(int(time.time()))
sig = "sha256=" + hmac.new(SECRET.encode(), f"{ts}.".encode() + payload, hashlib.sha256).hexdigest()
headers = {
    "Content-Type": "application/json",
    "X-Webhook-Timestamp": ts,
    "X-Webhook-Signature": sig,
    "X-Webhook-Delivery-Id": "dup-demo-1",
}

for i in range(3):
    r = requests.post("http://127.0.0.1:5006/hook", data=payload, headers=headers, timeout=5)
    body = r.json()
    print(f"  delivery {i+1}: HTTP {r.status_code}  duplicate={body.get('duplicate')}")

print(f"\n📦 Orders shipped: {len(orders_shipped)}  (webhook delivered 3 times)")

assert len(orders_shipped) == 1, (
    f"the order shipped {len(orders_shipped)} times -- the handler is NOT "
    f"idempotent, and every vendor retry is a duplicate charge"
)
print("✅ Three deliveries, one shipment. That is what idempotent means.")
print("\n⚠️  Our ledger is an in-memory dict, so it forgets on restart. In")
print("   production it is a UNIQUE constraint on delivery_id in the same")
print("   database transaction as the side effect -- otherwise a crash between")
print("   'ship' and 'record' leaves you exposed to exactly this bug.")

## 🆚 Step 7 — webhooks vs polling: the math

Imagine you integrate with a payment provider. You expect **~100 payments an
hour**, but you want to react within a few seconds of each one.


In [ ]:
payments_per_hour = 100
seconds_in_hour = 3600

# Option A: poll every 5 seconds
poll_interval = 5
polls_per_hour = seconds_in_hour / poll_interval
useful_polls = payments_per_hour          # optimistic: assume no two
                                          # payments share a poll window, so
                                          # this is an UPPER bound on useful
                                          # polls (and wasted is a lower bound)
wasted_polls = polls_per_hour - useful_polls

print("Polling every 5s")
print(f"  total polls / hour : {polls_per_hour:,.0f}")
print(f"  useful polls       : {useful_polls}")
print(f"  wasted polls       : {wasted_polls:,.0f} ({wasted_polls/polls_per_hour:.0%})")
print(f"  worst-case latency : {poll_interval}s")

print("\nWebhooks")
print(f"  total requests / h : {payments_per_hour}   # 1 per event")
print(f"  wasted requests    : 0")
print(f"  worst-case latency : ~network RTT")

print(f"\n📉 {polls_per_hour/payments_per_hour:.1f}x fewer requests, and the latency drops from")
print(f"   {poll_interval}s worst case to a single round trip.")
print("   The honest catch: those 100 webhook requests are inbound writes you")
print("   must authenticate and dedupe (Steps 5 and 6b), whereas the 720 polls")
print("   were your own outbound reads. You trade request volume for a public")
print("   attack surface -- and for a reconcile job to cover deliveries the")
print("   vendor gave up on.")

assert wasted_polls == polls_per_hour - useful_polls
assert polls_per_hour > payments_per_hour, "polling only loses when it out-paces events"


## ⚠️ Gotchas beginners hit

| Gotcha | What to do |
|---|---|
| Anyone can POST to your URL | **Always verify the signature** — `verify_signature()` in Step 2, attacked in Step 5. |
| A captured request replayed later | Sign a **timestamp** with the body and reject stale ones (Step 5, attack #4). |
| Same event delivered twice (retries) | Make handlers **idempotent** — dedupe on `delivery_id` (Step 6b). Answer **200**, not 409, or the vendor keeps retrying. |
| Your receiver is slow → vendor times out & retries | **Return 200 immediately**, push the work onto a queue. |
| Events arrive out of order | Include a timestamp/sequence in the payload and ignore older ones. |
| Localhost isn't reachable from the internet | Use a tunnel (`ngrok`, `cloudflared`) or poll during dev. |
| Receiver is down during an event | Vendor retries for a bit, then **gives up** — add a periodic reconcile job as backup. |


## 🎯 When to reach for webhooks

**Great fit**
- Infrequent-but-important events: payments, signups, CI builds, deploys.
- Cross-organisation integrations (you don't control both sides).
- Anywhere you'd otherwise set up a cron job that polls for changes.

**Bad fit**
- Your users are browsers — they don't have public URLs. Use SSE/WebSockets.
- Very high event rates (>100/s per subscriber) — batch into a stream
  (Kafka, Kinesis) instead.
- You need strict ordering / exactly-once — webhooks are at-least-once.


## 🧪 Quick quiz

1. You receive the same `order.paid` webhook twice. What did the vendor most likely do, and how should your handler behave?
2. Why is `hmac.compare_digest` preferred over `==` when comparing signatures?
3. Your receiver takes 30s to process an event. The vendor times out at 10s and retries. What's the cheapest fix?


In [ ]:
print("📝 Answers")
print("="*50)
print("1. It retried because an earlier attempt failed (or the ACK was lost")
print("   on the way back -- your handler may already have run!).")
print("   Dedupe by delivery_id and return 200 again. Step 6b showed three")
print("   identical deliveries producing exactly one shipment.")
print("")
print("2. compare_digest is constant-time so attackers can't guess a")
print("   signature byte-by-byte using timing differences.")
print("")
print("3. Acknowledge FAST: return 200 immediately, push work to a queue")
print("   (Redis/SQS/etc) and process it asynchronously. Note this makes")
print("   idempotency MORE important, not less -- the 200 no longer means")
print("   'done', so a retry can race the job you already queued.")

## 📚 Summary

- A **webhook** is an HTTP POST from a vendor to *your* URL when an event
  happens — the **inverse of polling**.
- You almost always want: **signature verification over a signed timestamp**
  (forgery *and* replay), **idempotent handlers keyed on `delivery_id`**,
  **fast ACK + async processing**, and a **reconciliation fallback** for the
  deliveries the vendor eventually gives up on.
- The receiver in this notebook implements all of those, and the cells attack
  it to prove each one fires. A receiver without them is not "a simplified
  example" — it is a public, unauthenticated write endpoint.
- Webhooks shine for low-to-medium-frequency server-to-server events. For
  browser clients you still want SSE or WebSockets (notebooks 4 and 5).
